In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import pickle
import random

In [2]:
# okay, let's construct a gnomAD hail table query to get the number of benign variants per gene for a given set of genes

In [3]:
import hail as hl
print("Hail version:", hl.version())
import subprocess, os

subprocess.run(["java", "-version"])
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

hl.init(
    log="/tmp/hail_test.log",
    spark_conf={
        "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
        "spark.hadoop.fs.s3a.aws.credentials.provider": "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider",
        "spark.hadoop.fs.s3a.endpoint": "s3.us-east-1.amazonaws.com",
    }
)

hl.default_reference("GRCh38")
hl.utils.range_table(10).show()

ht = hl.read_table(
    "s3a://gnomad-public-us-east-1/release/4.1.1/ht/browser/gnomad.browser.v4.1.1.sites.ht"
)

ht.describe()

Loading BokehJS ...

Hail version: 0.2.138-58956ebc28fc
JAVA_HOME: /usr/lib/jvm/java-11


openjdk version "11.0.25" 2024-10-15 LTS
OpenJDK Runtime Environment (Red_Hat-11.0.25.0.9-1) (build 11.0.25+9-LTS)
OpenJDK 64-Bit Server VM (Red_Hat-11.0.25.0.9-1) (build 11.0.25+9-LTS, mixed mode, sharing)


2026-07-31 11:51:09.357 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-07-31 11:51:13.070 WARN  SparkConf:72 - Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


Running on Apache Spark version 3.5.8
SparkUI available at http://gpudev1.wynton.ucsf.edu:4040
2026-07-31 11:51:15.820 SparkBackend$: WARN: This Hail JAR was compiled for Spark 3.5.3, running with Spark 3.5.8.
  Compatibility is not guaranteed.
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.138-58956ebc28fc
LOGGING: writing to /tmp/hail_test.log


""
idx
int32
0
1
2
3
4
5
6
7


2026-07-31 11:51:22.433 MetricsConfig: WARN: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


----------------------------------------
Global fields:
    'mane_select_version': str 
    'freq_meta': array<dict<str, str>> 
    'freq_index_dict': dict<str, int32> 
    'freq_meta_sample_count': array<int32> 
    'faf_meta': array<dict<str, str>> 
    'faf_index_dict': dict<str, int32> 
    'age_distribution': struct {
        bin_edges: array<float64>, 
        bin_freq: array<int32>, 
        n_smaller: int32, 
        n_larger: int32
    } 
    'downsamplings': dict<str, array<int32>> 
    'filtering_model': struct {
        filter_name: str, 
        score_name: str, 
        snv_cutoff: struct {
            bin: int32, 
            min_score: float64
        }, 
        indel_cutoff: struct {
            bin: int32, 
            min_score: float64
        }, 
        snv_training_variables: array<str>, 
        indel_training_variables: array<str>
    } 
    'inbreeding_coeff_cutoff': float64 
    'interval_qc_parameters': struct {
        per_platform: bool, 
        all_plat

In [4]:
ht.exome.freq.all.describe()

--------------------------------------------------------
Type:
        struct {
        ac: int32, 
        ac_raw: int32, 
        an: int32, 
        hemizygote_count: int32, 
        homozygote_count: int64, 
        ancestry_groups: array<struct {
            id: str, 
            ac: int32, 
            an: int32, 
            hemizygote_count: int32, 
            homozygote_count: int64
        }>
    }
--------------------------------------------------------
Source:
Index:
    ['row']
--------------------------------------------------------


In [4]:
import numpy as np
import pandas as pd
import os
import pickle

In [6]:
# load D&D genes
dnd = pd.read_csv('/wynton/group/capra/projects/dnd_project_results/RUN_04_23_26/filtered_transcripts/filtered_exon_info.csv')
dnd_genes=set(dnd.hgnc_symbol)
print(len(dnd_genes))

593


/scratch/gramey02/ipykernel_3488975/1631629579.py:2: DtypeWarning:

Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.



In [7]:
print(dnd.chromosome_name.unique())

['1' '17' '11' '7' '19' '4' '8' '12' '5' '6' '2' '10' '3' '18' '13' '16'
 '14' '9' '15' '22' '20' 'X' '21' 19 7 6 12 20 17 4 9 11 14 1 13 3 2 10 16
 15 22 8]


## Great. Let's write a query for the hail table to identify how many benign variants typically occur in these genes.

In [ ]:
# Your gene set as a Hail-usable literal
dnd_set = hl.literal(dnd_genes)

# Step 1: keep only variants where a canonical transcript maps to one of your genes
ht_genes = ht.filter(
    hl.any(
        lambda tc: tc.is_canonical & dnd_set.contains(tc.gene_symbol),
        ht.transcript_consequences
    )
)

print(ht_genes.count())  # sanity check — should shrink a lot from full table

In [ ]:
# Step 2: explode to one row per (variant, canonical transcript-in-your-geneset)
# so you get a gene_symbol column to group by later
ht_exp = ht_genes.annotate(
    tc_match = hl.filter(
        lambda tc: tc.is_canonical & dnd_set.contains(tc.gene_symbol),
        ht_genes.transcript_consequences
    )
)
ht_exp = ht_exp.explode(ht_exp.tc_match, name='tc')

In [ ]:
# Step 3: flag "benign" using clin_sig from exome + genome VEP colocated_variants
def has_benign(cv_array):
    return hl.or_else(
        hl.any(
            lambda cv: hl.any(lambda cs: cs.lower().contains('benign'), cv.clin_sig),
            cv_array
        ),
        False
    )

ht_exp = ht_exp.annotate(
    is_benign = has_benign(ht_exp.exome.vep115.colocated_variants) |
                has_benign(ht_exp.genome.vep115.colocated_variants)
)

In [ ]:
# Step 4: flag "observed" using AC > 0 across exome + genome
ht_exp = ht_exp.annotate(
    ac_total = hl.or_else(ht_exp.exome.freq.all.ac, 0) +
               hl.or_else(ht_exp.genome.freq.all.ac, 0)
)
ht_exp = ht_exp.annotate(is_observed = ht_exp.ac_total > 0)

In [ ]:
# Step 5: filter and count
ht_final = ht_exp.filter(ht_exp.is_benign & ht_exp.is_observed)

# per-gene counts
counts_by_gene = ht_final.group_by(ht_final.tc.gene_symbol).aggregate(
    n_benign_observed = hl.agg.count()
)
counts_by_gene.show(600)

# total across all 593 genes
total = ht_final.count()
print("Total observed benign variants across gene set:", total)

In [6]:
# Quick sanity check on a random sample first (fast, cheap)
sample = ht.sample(0.001, seed=1)

sample_exp_exome = sample.explode(sample.exome.vep115.colocated_variants)
sample_exp_genome = sample.explode(sample.genome.vep115.colocated_variants)

exome_clin_sig = sample_exp_exome.explode(sample_exp_exome.exome.vep115.colocated_variants.clin_sig)
genome_clin_sig = sample_exp_genome.explode(sample_exp_genome.genome.vep115.colocated_variants.clin_sig)

exome_vals = exome_clin_sig.aggregate(hl.agg.collect_as_set(exome_clin_sig.exome.vep115.colocated_variants.clin_sig))
genome_vals = genome_clin_sig.aggregate(hl.agg.collect_as_set(genome_clin_sig.genome.vep115.colocated_variants.clin_sig))

print("Exome clin_sig values:", exome_vals)
print("Genome clin_sig values:", genome_vals)

2026-07-29 11:01:49.219 DAGScheduler: WARN: Broadcasting large task binary with size 1908.4 KiB
2026-07-29 11:01:49.266 TaskSetManager: WARN: Stage 3 contains a task of very large size (1916 KiB). The maximum recommended task size is 1000 KiB.
2026-07-29 11:26:49.763 DAGScheduler: WARN: Broadcasting large task binary with size 1908.9 KiB
2026-07-29 11:26:49.786 TaskSetManager: WARN: Stage 6 contains a task of very large size (1916 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Exome clin_sig values: {'not_provided', 'association', 'likely_pathogenic', 'benign', 'pathogenic', 'likely_benign', 'uncertain_significance', 'uncertain_risk_allele'}
Genome clin_sig values: {'not_provided', 'association', 'likely_pathogenic', 'benign', 'pathogenic', 'likely_benign', 'uncertain_significance'}


In [ ]:
'not_provided', 'association', 'likely_pathogenic', 'benign', 'pathogenic', 'likely_benign', 'uncertain_significance'

In [9]:
# Let's run the first filter, stage 1
# filters to just high-confidence LOF variants among dnd genes
# load D&D genes
dnd = pd.read_csv('/wynton/group/capra/projects/dnd_project_results/RUN_04_23_26/filtered_transcripts/filtered_exon_info.csv')
dnd_genes = set(dnd.hgnc_symbol)
print(len(dnd_genes))

dnd_set = hl.literal(dnd_genes)

# combined predicate: canonical transcript, in gene set, LOFTEE high-confidence
def is_target_tc(tc):
    return (tc.is_canonical
            & dnd_set.contains(tc.gene_symbol)
            & (tc.lof == 'HC'))

# Stage 1: filter to variants with at least one matching transcript
ht_lof = ht.filter(
    hl.any(is_target_tc, ht.transcript_consequences)
)

print(ht_lof.count())  # sanity check — should be noticeably smaller than gene-only filter

/scratch/gramey02/ipykernel_3440661/3894790588.py:4: DtypeWarning:

Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.



593


2026-07-30 11:33:50.788 DAGScheduler: WARN: Broadcasting large task binary with size 1882.7 KiB
2026-07-30 11:33:50.888 TaskSetManager: WARN: Stage 3 contains a task of very large size (1890 KiB). The maximum recommended task size is 1000 KiB.
[Stage 3:====>                                                (807 + 56) / 9694]

KeyboardInterrupt: 

[Stage 3:====>                                                (816 + 56) / 9694]

In [7]:
# TEMPORARY: single-gene test set instead of full dnd_genes
dnd_set = hl.literal(dnd_genes)
test_gene_set = hl.literal({'SCN1A'})  # pick any gene you know is in dnd_genes and expect to have LoF variants

def is_target_tc_test(tc):
    return (tc.is_canonical
            & test_gene_set.contains(tc.gene_symbol)
            & (tc.lof == 'HC'))

ht_lof_test = ht.filter(
    hl.any(is_target_tc_test, ht.transcript_consequences)
)

print(ht_lof_test.count())
ht_lof_test.show(5)

2026-07-30 11:37:29.621 DAGScheduler: WARN: Broadcasting large task binary with size 1882.7 KiB
2026-07-30 11:37:29.661 TaskSetManager: WARN: Stage 3 contains a task of very large size (1890 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

104


[Stage 7:======>                                                (32 + 56) / 256]

KeyboardInterrupt: 

[Stage 7:=======>                                               (33 + 56) / 256]

In [10]:
# let's get the canonical transcript bounds for each gene
dnd.columns

Index(['Unnamed: 0', 'ensembl_gene_id', 'ensembl_transcript_id', 'hgnc_symbol',
       'chromosome_name', 'start_position', 'end_position', 'strand',
       'transcript_start', 'transcript_end', 'transcription_start_site',
       'transcript_length', 'transcript_is_canonical', 'transcript_tsl',
       'gene_biotype', 'transcript_biotype', 'ensembl_peptide_id',
       'exon_chrom_start', 'exon_chrom_end', 'ensembl_exon_id',
       'is_constitutive', 'rank', 'genomic_coding_start', 'genomic_coding_end',
       'cds_start', 'cds_end'],
      dtype='object')

In [13]:
canonical = dnd[dnd['transcript_is_canonical']==True]
canonical.hgnc_symbol.nunique()

521

In [15]:
gene_spans = dnd.groupby('hgnc_symbol').agg(
    chrom=('chromosome_name', 'first'),
    start=('start_position', 'min'),
    end=('end_position', 'max')
).reset_index()

In [16]:
gene_spans

,hgnc_symbol,chrom,start,end
0,AARS1,16,70251983,70289707
1,ABCB6,2,219209766,219219012
2,ABCC6,16,16149565,16223637
3,ABCC8,11,17392498,17476894
4,ABCC9,12,21797389,21942543
...,...,...,...,...
588,ZIC1,3,147393422,147510293
589,ZMIZ1,10,79068929,79316550
590,ZMYND8,20,47209214,47357804
591,ZNF292,6,87151803,87265943


In [19]:
'AARS1' in gene_spans.hgnc_symbol.values

True

In [21]:
row = gene_spans[gene_spans.hgnc_symbol == 'AARS1'].iloc[0]

chrom_str = str(row.chrom)
if not chrom_str.startswith('chr'):
    chrom_str = 'chr' + chrom_str

interval = hl.parse_locus_interval(
    f"{chrom_str}:{row.start}-{row.end}",
    reference_genome='GRCh38'
)

ht_region = hl.filter_intervals(ht, [interval])
print(ht_region.count())

21372


In [23]:
def is_target_tc_test(tc):
    return (tc.is_canonical
            & (tc.gene_symbol == 'AARS1')
            & (tc.lof == 'HC'))

ht_lof_test = ht_region.filter(hl.any(is_target_tc_test, ht_region.transcript_consequences))
print(ht_lof_test.count())

[Stage 5:=============================>                             (1 + 1) / 2]

192


In [25]:
# visualize results
df_test = ht_lof_test.select(
    ac = ht_lof_test.exome.freq.all.ac,
    an = ht_lof_test.exome.freq.all.an,
    hom = ht_lof_test.exome.freq.all.homozygote_count,
).to_pandas()

df_test

,locus,alleles,ac,an,hom
0,chr16:70252721,"[TC, T]",1,1461730,0
1,chr16:70252735,"[C, CG]",2,1461768,0
2,chr16:70252746,"[A, AG]",<NA>,<NA>,<NA>
3,chr16:70252783,"[G, A]",1,1461830,0
4,chr16:70252793,"[AACGTTCTTGCCTGTGGCCTGTGCAG, A]",1,1461860,0
...,...,...,...,...,...
187,chr16:70282710,"[GAAGA, G]",1,1461862,0
188,chr16:70282727,"[G, A]",4,1461838,0
189,chr16:70282730,"[G, A]",2,1461852,0
190,chr16:70282748,"[TTAGAG, T]",2,1461846,0


In [27]:
row_check = ht_lof_test.filter(
    ht_lof_test.locus == hl.locus('chr16', 70252746, reference_genome='GRCh38')
)

row_check.select(
    exome_ac = row_check.exome.freq.all.ac,
    genome_ac = row_check.genome.freq.all.ac,
).show()

,,,
locus,alleles,exome_ac,genome_ac
locus<GRCh38>,array<str>,int32,int32
chr16:70252746,"[""A"",""AG""]",NA,1


In [30]:
# great, now let's filter this table down to benign heterozygous variants
# Stage 2a: compute total AC, homozygote count, hemizygote count across exome + genome
ht_lof_test = ht_lof_test.annotate(
    ac_total = hl.or_else(ht_lof_test.exome.freq.all.ac, 0) +
               hl.or_else(ht_lof_test.genome.freq.all.ac, 0),
    hom_total = hl.or_else(ht_lof_test.exome.freq.all.homozygote_count, hl.int64(0)) +
                hl.or_else(ht_lof_test.genome.freq.all.homozygote_count, hl.int64(0)),
    hemi_total = hl.or_else(ht_lof_test.exome.freq.all.hemizygote_count, 0) +
                 hl.or_else(ht_lof_test.genome.freq.all.hemizygote_count, 0),
)

# Stage 2b: derive heterozygote count
ht_lof_test = ht_lof_test.annotate(
    het_total = ht_lof_test.ac_total - (2 * ht_lof_test.hom_total) - ht_lof_test.hemi_total
)

# Stage 2c: flag whether this gene is chrX (from your dnd file, not locus-based)
is_chrX_gene = False #'AARS1' in chrX_genes   # plain Python bool, since we're testing one gene at a time

# Stage 2d: filter to heterozygous-observed (or hemizygous-observed, if chrX gene)
if is_chrX_gene:
    ht_het_test = ht_lof_test.filter(ht_lof_test.hemi_total > 0)
else:
    ht_het_test = ht_lof_test.filter(ht_lof_test.het_total > 0)

print(ht_het_test.count())

[Stage 9:=============================>                             (1 + 1) / 2]

170


In [31]:
ht_het_test.show()

+----------------+------------------------------------+
| locus          | alleles                            |
+----------------+------------------------------------+
| locus<GRCh38>  | array<str>                         |
+----------------+------------------------------------+
| chr16:70252721 | ["TC","T"]                         |
| chr16:70252735 | ["C","CG"]                         |
| chr16:70252746 | ["A","AG"]                         |
| chr16:70252783 | ["G","A"]                          |
| chr16:70252793 | ["AACGTTCTTGCCTGTGGCCTGTGCAG","A"] |
| chr16:70252813 | ["G","A"]                          |
| chr16:70252818 | ["GAC","G"]                        |
| chr16:70252834 | ["C","CT"]                         |
| chr16:70252864 | ["G","A"]                          |
| chr16:70252871 | ["C","T"]                          |
+----------------+------------------------------------+

+-------------------------------+
| exome.vep115.allele_string    |
+-------------------------------+
| str                           |
+-------------------------------+
| "C/-"                         |
| "-/G"                         |
| NA                            |
| "G/A"                         |
| "ACGTTCTTGCCTGTGGCCTGTGCAG/-" |
| "G/A"                         |
| "AC/-"                        |
| "-/T"                         |
| "G/A"                         |
| "C/T"                         |
+-------------------------------+

+------------------------------------------------------------------------------+
| exome.vep115.colocated_variants                                              |
+------------------------------------------------------------------------------+
| array<struct{allele_string: str, clin_sig: array<str>, clin_sig_allele: s... |
+------------------------------------------------------------------------------+
| NA                                                                           |
| NA                                                                           |
| NA                                                                           |
| NA                                                                           |
| NA                                                                           |
| NA                                                                           |
| NA                                                                           |
| NA                                                                           |
| NA                                                                           |
| [("C/T",["uncertain_significance"],"T:uncertain_significance",70252871,"r... |
+------------------------------------------------------------------------------+

+------------------+-----------------+
| exome.vep115.end | exome.vep115.id |
+------------------+-----------------+
|            int32 | str             |
+------------------+-----------------+
|         70252722 | "."             |
|         70252735 | "."             |
|               NA | NA              |
|         70252783 | "."             |
|         70252818 | "."             |
|         70252813 | "."             |
|         70252820 | "."             |
|         70252834 | "."             |
|         70252864 | "."             |
|         70252871 | "."             |
+------------------+-----------------+

+--------------------------------------------------------+
| exome.vep115.input                                     |
+--------------------------------------------------------+
| str                                                    |
+--------------------------------------------------------+
| "chr16	70252721	.	TC	T	.	.	GT"                         |
| "chr16	70252735	.	C	CG	.	.	GT"                         |
| NA                                                     |
| "chr16	70252783	.	G	A	.	.	GT"                          |
| "chr16	70252793	.	AACGTTCTTGCCTGTGGCCTGTGCAG	A	.	.	GT" |
| "chr16	70252813	.	G	A	.	.	GT"               

In [32]:
df_het_test = ht_het_test.select(
    ac_total = ht_het_test.ac_total,
    hom_total = ht_het_test.hom_total,
    hemi_total = ht_het_test.hemi_total,
    het_total = ht_het_test.het_total,
).to_pandas()

df_het_test

,locus,alleles,ac_total,hom_total,hemi_total,het_total
0,chr16:70252721,"[TC, T]",1,0,0,1
1,chr16:70252735,"[C, CG]",2,0,0,2
2,chr16:70252746,"[A, AG]",1,0,0,1
3,chr16:70252783,"[G, A]",1,0,0,1
4,chr16:70252793,"[AACGTTCTTGCCTGTGGCCTGTGCAG, A]",1,0,0,1
...,...,...,...,...,...,...
165,chr16:70282710,"[GAAGA, G]",1,0,0,1
166,chr16:70282727,"[G, A]",11,0,0,11
167,chr16:70282730,"[G, A]",2,0,0,2
168,chr16:70282748,"[TTAGAG, T]",3,0,0,3


In [33]:
# finally, get clinical classifications
# Stage 3: classify each variant into a clinical significance category

CATEGORIES = ['pathogenic', 'likely_pathogenic', 'uncertain_significance',
              'likely_benign', 'benign', 'association', 'not_provided']

def clinsig_terms(cv_array):
    return hl.or_else(
        hl.flatmap(lambda cv: cv.clin_sig, cv_array),
        hl.empty_array(hl.tstr)
    ).map(lambda s: s.lower())

ht_het_test = ht_het_test.annotate(
    clinsig_terms = clinsig_terms(ht_het_test.exome.vep115.colocated_variants).extend(
                    clinsig_terms(ht_het_test.genome.vep115.colocated_variants))
)

ht_het_test = ht_het_test.annotate(
    clinsig_category = (
        hl.case()
        .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('pathogenic') & ~s.contains('likely')), 'pathogenic')
        .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('likely_pathogenic')), 'likely_pathogenic')
        .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('uncertain')), 'uncertain_significance')
        .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('likely_benign')), 'likely_benign')
        .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('benign') & ~s.contains('likely')), 'benign')
        .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('association')), 'association')
        .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('not_provided')), 'not_provided')
        .default('other_or_unclassified')
    )
)

# tally counts per category for this one gene
ht_het_test.group_by(ht_het_test.clinsig_category).aggregate(n = hl.agg.count()).show()

,
clinsig_category,n
str,int64
"""likely_pathogenic""",7
"""other_or_unclassified""",131
"""pathogenic""",25
"""uncertain_significance""",7


In [35]:
ht_unclassified = ht_het_test.filter(ht_het_test.clinsig_category == 'other_or_unclassified')

ht_unclassified.select(
    ht_unclassified.clinsig_terms
).show(20)

,,
locus,alleles,clinsig_terms
locus<GRCh38>,array<str>,array<str>
chr16:70252721,"[""TC"",""T""]",[]
chr16:70252735,"[""C"",""CG""]",[]
chr16:70252746,"[""A"",""AG""]",[]
chr16:70252783,"[""G"",""A""]",[]
chr16:70252793,"[""AACGTTCTTGCCTGTGGCCTGTGCAG"",""A""]",[]
chr16:70252813,"[""G"",""A""]",[]
chr16:70252818,"[""GAC"",""G""]",[]
chr16:70252834,"[""C"",""CT""]",[]


In [36]:
ht_unclassified.aggregate(
    hl.agg.counter(hl.len(ht_unclassified.clinsig_terms) == 0)
)

{True: 131}

In [ ]:
# let's show the results

In [ ]:
## Okay, now let's combine it all into one big code section that we can parallelize across genes


In [4]:
# define input variables
dnd_file='/wynton/group/capra/projects/dnd_project_results/RUN_04_23_26/filtered_transcripts/filtered_exon_info.csv'
output_file='/wynton/home/capra/gramey02/dnd_project/revisions/results/dnd_lof_clinsig_counts.csv'
checkpoint_file='/wynton/home/capra/gramey02/dnd_project/revisions/checkpoints/dnd_lof_clinsig_checkpoint.pkl'

In [6]:
hl.stop()

In [7]:
# set up hail table params
hl.init(
    log="/tmp/hail_test.log",
    spark_conf={
        "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
        "spark.hadoop.fs.s3a.aws.credentials.provider": "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider",
        "spark.hadoop.fs.s3a.endpoint": "s3.us-east-1.amazonaws.com",
    }
)
hl.default_reference("GRCh38")
hl.utils.range_table(10).show()

# load main hail table
ht = hl.read_table(
    "s3a://gnomad-public-us-east-1/release/4.1.1/ht/browser/gnomad.browser.v4.1.1.sites.ht"
)



2026-07-30 15:19:26.299 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-07-30 15:19:27.607 WARN  SparkConf:72 - Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


Running on Apache Spark version 3.5.8
SparkUI available at http://gpudev1.wynton.ucsf.edu:4040
2026-07-30 15:19:29.274 SparkBackend$: WARN: This Hail JAR was compiled for Spark 3.5.3, running with Spark 3.5.8.
  Compatibility is not guaranteed.
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.138-58956ebc28fc
LOGGING: writing to /tmp/hail_test.log


""
idx
int32
0
1
2
3
4
5
6
7


2026-07-30 15:19:34.403 MetricsConfig: WARN: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


TypeError: 'set' object is not subscriptable

In [9]:
def is_target_tc_test(tc, gene):
    return (tc.is_canonical
            & (tc.gene_symbol == gene)
            & (tc.lof == 'HC'))

In [10]:
# load D&D genes
#dnd_file=args.dnd_file
#'/wynton/group/capra/projects/dnd_project_results/RUN_04_23_26/filtered_transcripts/filtered_exon_info.csv'
dnd = pd.read_csv(dnd_file, dtype={'chromosome_name':'str'})
dnd_genes=set(dnd.hgnc_symbol)

# create a hail literal set of dnd genes
#dnd_set = hl.literal(dnd_genes)
dnd_set=set(['NEFL','AARS1'])
# separate out a subset of chromosome X genes
chrX_genes = set(dnd[dnd['chromosome_name']=='X']['hgnc_symbol'].unique())

# filtering genes by locus is fast relative to filtering by gene name. get loci below
gene_spans = dnd.groupby('hgnc_symbol').agg(
    chrom=('chromosome_name', 'first'),
    start=('start_position', 'min'),
    end=('end_position', 'max')
).reset_index()
gene_spans = gene_spans[gene_spans['hgnc_symbol'].isin(['NEFL','AARS1'])]

results={}
# loop over each gene
for i,r in gene_spans.iterrows():
    gene=r.hgnc_symbol
    try:
        # filter for the current gene's locus
        chrom_str = str(r.chrom)
        if not chrom_str.startswith('chr'):
            chrom_str = 'chr' + chrom_str
        interval = hl.parse_locus_interval(
            f"{chrom_str}:{r.start}-{r.end}",
            reference_genome='GRCh38'
        )
        ht_region = hl.filter_intervals(ht, [interval])
        
        # filter to high-confidence loss-of-function variants
        ht_lof_test = ht_region.filter(
            hl.any(lambda tc: is_target_tc_test(tc, gene), ht_region.transcript_consequences)
        )

        # compute total AC, homozygote count, hemizygote count across exome + genome
        ht_lof_test = ht_lof_test.annotate(
            ac_total = hl.or_else(ht_lof_test.exome.freq.all.ac, 0) +
                    hl.or_else(ht_lof_test.genome.freq.all.ac, 0),
            hom_total = hl.or_else(ht_lof_test.exome.freq.all.homozygote_count, hl.int64(0)) +
                        hl.or_else(ht_lof_test.genome.freq.all.homozygote_count, hl.int64(0)),
            hemi_total = hl.or_else(ht_lof_test.exome.freq.all.hemizygote_count, 0) +
                        hl.or_else(ht_lof_test.genome.freq.all.hemizygote_count, 0),
        )

        # derive heterozygote count
        ht_lof_test = ht_lof_test.annotate(
            het_total = ht_lof_test.ac_total - (2 * ht_lof_test.hom_total) - ht_lof_test.hemi_total
        )

        # flag whether this gene is chrX (from your dnd file, not locus-based)
        is_chrX_gene = gene in chrX_genes   # plain Python bool, since we're testing one gene at a time

        # Stage 2d: filter to heterozygous-observed (or hemizygous-observed, if chrX gene)
        if is_chrX_gene:
            ht_het_test = ht_lof_test.filter(ht_lof_test.hemi_total > 0)
        else:
            ht_het_test = ht_lof_test.filter(ht_lof_test.het_total > 0)

        # classify each variant into a clinical significance category

        CATEGORIES = ['pathogenic', 'likely_pathogenic', 'uncertain_significance',
                    'likely_benign', 'benign', 'association', 'not_provided']

        def clinsig_terms(cv_array):
            return hl.or_else(
                hl.flatmap(lambda cv: cv.clin_sig, cv_array),
                hl.empty_array(hl.tstr)
            ).map(lambda s: s.lower())

        ht_het_test = ht_het_test.annotate(
            clinsig_terms = clinsig_terms(ht_het_test.exome.vep115.colocated_variants).extend(
                            clinsig_terms(ht_het_test.genome.vep115.colocated_variants))
        )

        ht_het_test = ht_het_test.annotate(
            clinsig_category = (
                hl.case()
                .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('pathogenic') & ~s.contains('likely')), 'pathogenic')
                .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('likely_pathogenic')), 'likely_pathogenic')
                .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('uncertain')), 'uncertain_significance')
                .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('likely_benign')), 'likely_benign')
                .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('benign') & ~s.contains('likely')), 'benign')
                .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('association')), 'association')
                .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('not_provided')), 'not_provided')
                .default('other_or_unclassified')
            )
        )

        # tally counts per category for this one gene and store in dictionary
        gene_counts_df = ht_het_test.group_by(ht_het_test.clinsig_category).aggregate(
            n = hl.agg.count()
        ).to_pandas()

        # build a dict with all 7 categories + the unclassified bucket, defaulting to 0
        ALL_CATEGORIES = CATEGORIES + ['other_or_unclassified']
        gene_counts_dict = {cat: 0 for cat in ALL_CATEGORIES}
        gene_counts_dict.update(dict(zip(gene_counts_df['clinsig_category'], gene_counts_df['n'])))

        results[gene] = gene_counts_dict

        # # save a checkpoint
        # if i % 50 == 0:  # every 50 genes
        #     with open(checkpoint_file, 'wb') as f:
        #         pickle.dump(results, f)
    except Exception as e:
        print(f"FAILED on gene {gene}: {e}")
        continue

In [11]:
results

{'AARS1': {'pathogenic': np.int64(25),
  'likely_pathogenic': np.int64(7),
  'uncertain_significance': np.int64(7),
  'likely_benign': 0,
  'benign': 0,
  'association': 0,
  'not_provided': 0,
  'other_or_unclassified': np.int64(131)},
 'NEFL': {'pathogenic': np.int64(11),
  'likely_pathogenic': np.int64(2),
  'uncertain_significance': np.int64(7),
  'likely_benign': 0,
  'benign': 0,
  'association': 0,
  'not_provided': 0,
  'other_or_unclassified': np.int64(49)}}

In [12]:
final_df = pd.DataFrame.from_dict(results, orient='index')
final_df.index.name = 'gene_symbol'

In [13]:
final_df

,pathogenic,likely_pathogenic,uncertain_significance,likely_benign,benign,association,not_provided,other_or_unclassified
gene_symbol,,,,,,,,
AARS1,25,7,7,0,0,0,0,131
NEFL,11,2,7,0,0,0,0,49


In [5]:
hl.stop()

In [6]:
# set up hail table params
hl.init(
    log="/tmp/hail_test.log",
    spark_conf={
        "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
        "spark.hadoop.fs.s3a.aws.credentials.provider": "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider",
        "spark.hadoop.fs.s3a.endpoint": "s3.us-east-1.amazonaws.com",
    }
)
hl.default_reference("GRCh38")
hl.utils.range_table(10).show()

# load main hail table
ht = hl.read_table(
    "s3a://gnomad-public-us-east-1/release/4.1.1/ht/browser/gnomad.browser.v4.1.1.sites.ht"
)

2026-07-31 11:51:45.965 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-07-31 11:51:46.965 WARN  SparkConf:72 - Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


Running on Apache Spark version 3.5.8
SparkUI available at http://gpudev1.wynton.ucsf.edu:4040
2026-07-31 11:51:48.658 SparkBackend$: WARN: This Hail JAR was compiled for Spark 3.5.3, running with Spark 3.5.8.
  Compatibility is not guaranteed.
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.138-58956ebc28fc
LOGGING: writing to /tmp/hail_test.log


""
idx
int32
0
1
2
3
4
5
6
7


2026-07-31 11:51:53.026 MetricsConfig: WARN: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [8]:
# build ALL gene intervals at once (not in a loop)
dnd_file='/wynton/group/capra/projects/dnd_project_results/RUN_04_23_26/filtered_transcripts/filtered_exon_info.csv'
dnd = pd.read_csv(dnd_file, dtype={'chromosome_name':'str'})
dnd_genes=set(dnd.hgnc_symbol)
# create a hail literal set of dnd genes
dnd_set = hl.literal(dnd_genes)
# separate out a subset of chromosome X genes
chrX_genes = set(dnd[dnd['chromosome_name']=='X']['hgnc_symbol'].unique())

# filtering genes by locus is fast relative to filtering by gene name. get loci below
gene_spans = dnd.groupby('hgnc_symbol').agg(
    chrom=('chromosome_name', 'first'),
    start=('start_position', 'min'),
    end=('end_position', 'max')
).reset_index()
intervals = []
for r in gene_spans.itertuples():
    chrom_str = str(r.chrom)
    if not chrom_str.startswith('chr'):
        chrom_str = 'chr' + chrom_str
    intervals.append(hl.parse_locus_interval(f"{chrom_str}:{r.start}-{r.end}", reference_genome='GRCh38'))

ht_pruned = hl.filter_intervals(ht, intervals)
ht_pruned.write('/wynton/home/capra/gramey02/dnd_project/revisions/hail_tables/gnomad_dnd_pruned.ht', overwrite=True)

In [ ]:
# Great, now the table is saved

In [ ]:


# load D&D genes
#dnd_file=args.dnd_file
#'/wynton/group/capra/projects/dnd_project_results/RUN_04_23_26/filtered_transcripts/filtered_exon_info.csv'
dnd = pd.read_csv(dnd_file, dtype={'chromosome_name':'str'})
dnd_genes=set(dnd.hgnc_symbol)

# create a hail literal set of dnd genes
dnd_set = hl.literal(dnd_genes)
# separate out a subset of chromosome X genes
chrX_genes = set(dnd[dnd['chromosome_name']=='X']['hgnc_symbol'].unique())

# filtering genes by locus is fast relative to filtering by gene name. get loci below
gene_spans = dnd.groupby('hgnc_symbol').agg(
    chrom=('chromosome_name', 'first'),
    start=('start_position', 'min'),
    end=('end_position', 'max')
).reset_index()

In [ ]:
# redo with all genes
# parse input args

# set up hail table params
hl.init(
    log="/tmp/hail_test.log",
    spark_conf={
        "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
        "spark.hadoop.fs.s3a.aws.credentials.provider": "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider",
        "spark.hadoop.fs.s3a.endpoint": "s3.us-east-1.amazonaws.com",
    }
)
hl.default_reference("GRCh38")
hl.utils.range_table(10).show()

# load main hail table
ht = hl.read_table(
    "s3a://gnomad-public-us-east-1/release/4.1.1/ht/browser/gnomad.browser.v4.1.1.sites.ht"
)

# load D&D genes
#dnd_file=args.dnd_file
#'/wynton/group/capra/projects/dnd_project_results/RUN_04_23_26/filtered_transcripts/filtered_exon_info.csv'
dnd = pd.read_csv(dnd_file, dtype={'chromosome_name':'str'})
dnd_genes=set(dnd.hgnc_symbol)

# create a hail literal set of dnd genes
dnd_set = hl.literal(dnd_genes)
# separate out a subset of chromosome X genes
chrX_genes = set(dnd[dnd['chromosome_name']=='X']['hgnc_symbol'].unique())

# filtering genes by locus is fast relative to filtering by gene name. get loci below
gene_spans = dnd.groupby('hgnc_symbol').agg(
    chrom=('chromosome_name', 'first'),
    start=('start_position', 'min'),
    end=('end_position', 'max')
).reset_index()

results={}
# loop over each gene
for i,r in gene_spans.iterrows():
    gene=r.hgnc_symbol
    try:
        # filter for the current gene's locus
        chrom_str = str(r.chrom)
        if not chrom_str.startswith('chr'):
            chrom_str = 'chr' + chrom_str
        interval = hl.parse_locus_interval(
            f"{chrom_str}:{r.start}-{r.end}",
            reference_genome='GRCh38'
        )
        ht_region = hl.filter_intervals(ht, [interval])
        
        # filter to high-confidence loss-of-function variants
        ht_lof_test = ht_region.filter(
            hl.any(lambda tc: is_target_tc_test(tc, gene), ht_region.transcript_consequences)
        )

        # compute total AC, homozygote count, hemizygote count across exome + genome
        ht_lof_test = ht_lof_test.annotate(
            ac_total = hl.or_else(ht_lof_test.exome.freq.all.ac, 0) +
                    hl.or_else(ht_lof_test.genome.freq.all.ac, 0),
            hom_total = hl.or_else(ht_lof_test.exome.freq.all.homozygote_count, hl.int64(0)) +
                        hl.or_else(ht_lof_test.genome.freq.all.homozygote_count, hl.int64(0)),
            hemi_total = hl.or_else(ht_lof_test.exome.freq.all.hemizygote_count, 0) +
                        hl.or_else(ht_lof_test.genome.freq.all.hemizygote_count, 0),
        )

        # derive heterozygote count
        ht_lof_test = ht_lof_test.annotate(
            het_total = ht_lof_test.ac_total - (2 * ht_lof_test.hom_total) - ht_lof_test.hemi_total
        )

        # flag whether this gene is chrX (from your dnd file, not locus-based)
        is_chrX_gene = gene in chrX_genes   # plain Python bool, since we're testing one gene at a time

        # Stage 2d: filter to heterozygous-observed (or hemizygous-observed, if chrX gene)
        if is_chrX_gene:
            ht_het_test = ht_lof_test.filter(ht_lof_test.hemi_total > 0)
        else:
            ht_het_test = ht_lof_test.filter(ht_lof_test.het_total > 0)

        # classify each variant into a clinical significance category

        CATEGORIES = ['pathogenic', 'likely_pathogenic', 'uncertain_significance',
                    'likely_benign', 'benign', 'association', 'not_provided']

        def clinsig_terms(cv_array):
            return hl.or_else(
                hl.flatmap(lambda cv: cv.clin_sig, cv_array),
                hl.empty_array(hl.tstr)
            ).map(lambda s: s.lower())

        ht_het_test = ht_het_test.annotate(
            clinsig_terms = clinsig_terms(ht_het_test.exome.vep115.colocated_variants).extend(
                            clinsig_terms(ht_het_test.genome.vep115.colocated_variants))
        )

        ht_het_test = ht_het_test.annotate(
            clinsig_category = (
                hl.case()
                .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('pathogenic') & ~s.contains('likely')), 'pathogenic')
                .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('likely_pathogenic')), 'likely_pathogenic')
                .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('uncertain')), 'uncertain_significance')
                .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('likely_benign')), 'likely_benign')
                .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('benign') & ~s.contains('likely')), 'benign')
                .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('association')), 'association')
                .when(ht_het_test.clinsig_terms.any(lambda s: s.contains('not_provided')), 'not_provided')
                .default('other_or_unclassified')
            )
        )

        # tally counts per category for this one gene and store in dictionary
        gene_counts_df = ht_het_test.group_by(ht_het_test.clinsig_category).aggregate(
            n = hl.agg.count()
        ).to_pandas()

        # build a dict with all 7 categories + the unclassified bucket, defaulting to 0
        ALL_CATEGORIES = CATEGORIES + ['other_or_unclassified']
        gene_counts_dict = {cat: 0 for cat in ALL_CATEGORIES}
        gene_counts_dict.update(dict(zip(gene_counts_df['clinsig_category'], gene_counts_df['n'])))

        results[gene] = gene_counts_dict

        # save a checkpoint
        if i % 50 == 0:  # every 50 genes
            with open(checkpoint_file, 'wb') as f:
                pickle.dump(results, f)
            print(i)
    except Exception as e:
        print(f"FAILED on gene {gene}: {e}")
        continue

# save results
final_df = pd.DataFrame.from_dict(results, orient='index')
final_df.index.name = 'gene_symbol'
final_df.to_csv(output_file)

2026-07-30 15:52:24.335 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-07-30 15:52:25.504 WARN  SparkConf:72 - Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).


Running on Apache Spark version 3.5.8
SparkUI available at http://gpudev1.wynton.ucsf.edu:4040
2026-07-30 15:52:27.207 SparkBackend$: WARN: This Hail JAR was compiled for Spark 3.5.3, running with Spark 3.5.8.
  Compatibility is not guaranteed.
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.138-58956ebc28fc
LOGGING: writing to /tmp/hail_test.log


""
idx
int32
0
1
2
3
4
5
6
7


...

400


450


2026-07-30 17:00:14.616 TorrentBroadcast: ERROR: Store broadcast broadcast_5527 fail, remove all pieces of the broadcast
2026-07-30 17:00:14.617 LoweringPipeline: ERROR: error while applying lowering 'LowerAndExecuteShuffles'
java.lang.OutOfMemoryError: Java heap space
	at java.nio.HeapByteBuffer.<init>(HeapByteBuffer.java:61) ~[?:?]
	at java.nio.ByteBuffer.allocate(ByteBuffer.java:348) ~[?:?]
	at org.apache.spark.broadcast.TorrentBroadcast$.$anonfun$blockifyObject$1(TorrentBroadcast.scala:360) ~[spark-core_2.12-3.5.8.jar:3.5.8]
	at org.apache.spark.broadcast.TorrentBroadcast$.$anonfun$blockifyObject$1$adapted(TorrentBroadcast.scala:360) ~[spark-core_2.12-3.5.8.jar:3.5.8]
	at org.apache.spark.broadcast.TorrentBroadcast$$$Lambda$2573/0x0000000840d6c840.apply(Unknown Source) ~[?:?]
	at org.apache.spark.util.io.ChunkedByteBufferOutputStream.allocateNewChunkIfNeeded(ChunkedByteBufferOutputStream.scala:87) ~[spark-core_2.12-3.5.8.jar:3.5.8]
	at org.apache.spark.util.io.ChunkedByteBufferOutp

FAILED on gene SMAD9: An error occurred while calling o124.close


2026-07-30 17:05:20.857 TorrentBroadcast: ERROR: Store broadcast broadcast_5552 fail, remove all pieces of the broadcast
2026-07-30 17:05:20.859 LoweringPipeline: ERROR: error while applying lowering 'LowerAndExecuteShuffles'
java.lang.OutOfMemoryError: Java heap space
2026-07-30 17:05:20.859 LoweringPipeline: ERROR: error while applying lowering 'EvalRelationalLets'
java.lang.OutOfMemoryError: Java heap space
2026-07-30 17:05:39.809 TorrentBroadcast: ERROR: Store broadcast broadcast_5555 fail, remove all pieces of the broadcast
2026-07-30 17:05:39.811 LoweringPipeline: ERROR: error while applying lowering 'LowerAndExecuteShuffles'
java.lang.OutOfMemoryError: Java heap space
2026-07-30 17:05:39.811 LoweringPipeline: ERROR: error while applying lowering 'EvalRelationalLets'
java.lang.OutOfMemoryError: Java heap space
2026-07-30 17:06:00.073 TorrentBroadcast: ERROR: Store broadcast broadcast_5558 fail, remove all pieces of the broadcast
2026-07-30 17:06:00.074 LoweringPipeline: ERROR: er

In [ ]:
# Final code to run before python script on compute node

# set up hail table params
hl.init(
    log="/tmp/hail_test.log",
    spark_conf={
        "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
        "spark.hadoop.fs.s3a.aws.credentials.provider": "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider",
        "spark.hadoop.fs.s3a.endpoint": "s3.us-east-1.amazonaws.com",
    }
)
hl.default_reference("GRCh38")
hl.utils.range_table(10).show()

# load main hail table
ht = hl.read_table(
    "s3a://gnomad-public-us-east-1/release/4.1.1/ht/browser/gnomad.browser.v4.1.1.sites.ht"
)

# build ALL gene intervals at once (not in a loop)
dnd_file='/wynton/group/capra/projects/dnd_project_results/RUN_04_23_26/filtered_transcripts/filtered_exon_info.csv'
dnd = pd.read_csv(dnd_file, dtype={'chromosome_name':'str'})
dnd_genes=set(dnd.hgnc_symbol)
# create a hail literal set of dnd genes
dnd_set = hl.literal(dnd_genes)
# separate out a subset of chromosome X genes
chrX_genes = set(dnd[dnd['chromosome_name']=='X']['hgnc_symbol'].unique())

# filtering genes by locus is fast relative to filtering by gene name. get loci below
gene_spans = dnd.groupby('hgnc_symbol').agg(
    chrom=('chromosome_name', 'first'),
    start=('start_position', 'min'),
    end=('end_position', 'max')
).reset_index()
intervals = []
for r in gene_spans.itertuples():
    chrom_str = str(r.chrom)
    if not chrom_str.startswith('chr'):
        chrom_str = 'chr' + chrom_str
    intervals.append(hl.parse_locus_interval(f"{chrom_str}:{r.start}-{r.end}", reference_genome='GRCh38'))

ht_pruned = hl.filter_intervals(ht, intervals)
ht_pruned.write('/wynton/home/capra/gramey02/dnd_project/revisions/hail_tables/gnomad_dnd_pruned.ht', overwrite=True)

In [10]:
ht_check = hl.read_table('/wynton/home/capra/gramey02/dnd_project/revisions/hail_tables/gnomad_dnd_pruned.ht')
print(ht_check.count())

20426780


In [11]:
ht_check.describe()

----------------------------------------
Global fields:
    'mane_select_version': str 
    'freq_meta': array<dict<str, str>> 
    'freq_index_dict': dict<str, int32> 
    'freq_meta_sample_count': array<int32> 
    'faf_meta': array<dict<str, str>> 
    'faf_index_dict': dict<str, int32> 
    'age_distribution': struct {
        bin_edges: array<float64>, 
        bin_freq: array<int32>, 
        n_smaller: int32, 
        n_larger: int32
    } 
    'downsamplings': dict<str, array<int32>> 
    'filtering_model': struct {
        filter_name: str, 
        score_name: str, 
        snv_cutoff: struct {
            bin: int32, 
            min_score: float64
        }, 
        indel_cutoff: struct {
            bin: int32, 
            min_score: float64
        }, 
        snv_training_variables: array<str>, 
        indel_training_variables: array<str>
    } 
    'inbreeding_coeff_cutoff': float64 
    'interval_qc_parameters': struct {
        per_platform: bool, 
        all_plat